# Part 3 — Risk gate

Order alone is not enough on a shared bench.

STEP-06 is high risk: fastening only if hands are clear.

Run from the repo root so imports resolve (`jupyter notebook notebooks/02_risk_gate.ipynb`).

## Rule

$$\mathrm{can\_start}(s) \iff \mathrm{Pred}(s) \subseteq C \land (\mathrm{risk}(s) \neq \mathrm{high} \lor H)$$

where $H$ = hands clear.

In [ ]:
import sys
from pathlib import Path

# repo root on path when notebook cwd is notebooks/
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))

from shopfloor import ShopContext, WorkOrder, can_start, complete_step, parse_sop

steps = parse_sop(root / "docs/sop_samples/miam_assembly_subset.md")
print("loaded", len(steps), "steps")
print("STEP-06", steps["STEP-06"])

In [ ]:
# STEP-01..05 done — order is fine for STEP-06
order = WorkOrder(
    id="WO-310",
    completed={f"STEP-{i:02d}" for i in range(1, 6)},
)
step6 = steps["STEP-06"]

ctx_busy = ShopContext(hands_clear=False)
ok, reasons = can_start(step6, order, ctx_busy)
print("hands busy:", ok, reasons)

ctx_clear = ShopContext(hands_clear=True)
ok, reasons = can_start(step6, order, ctx_clear)
print("hands clear:", ok, reasons)

In [ ]:
# walk until STEP-06, then clear hands
order = WorkOrder(id="WO-311")
ctx = ShopContext(hands_clear=False)

for sid in [f"STEP-{i:02d}" for i in range(1, 6)]:
    ok, reasons = complete_step(order, steps[sid], ctx)
    print(sid, ok)

ok, reasons = complete_step(order, step6, ctx)
print("STEP-06 blocked?", ok, reasons)

ctx.hands_clear = True
ok, reasons = complete_step(order, step6, ctx)
print("STEP-06 after clear:", ok, sorted(order.completed))

## Note

`is_allowed` = order only. `can_start` adds the safety gate.
Same idea as Part 1–2, just one more condition for high-risk steps.